# LangChain: Building LLM-Powered Apps
### A Deep Technical Blog — Code Companion Notebook

> **Tools:** Python · LangChain · OpenAI  
> **Purpose:** All code examples from the blog, in runnable form.  
> **Note:** Set your `OPENAI_API_KEY` before running any cell.

---

## ⚙️ Setup — Install Dependencies

In [ ]:
# Install all required packages
# Run this cell once before anything else
!pip install langchain langchain-openai langchain-community faiss-cpu duckduckgo-search -q

In [ ]:
import os

# Set your OpenAI API key here
os.environ["OPENAI_API_KEY"] = "your-api-key-here"

---
## Example 1 — Basic LLM Call

The most fundamental operation: initialize a Chat Model and send it a message.  
LangChain wraps the provider SDK so the interface stays the same across OpenAI, Anthropic, etc.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# Initialize the chat model with desired parameters
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# Send a message and get a response
response = llm.invoke([HumanMessage(content="What is machine learning?")])
print(response.content)

---
## Example 2 — PromptTemplate Usage

PromptTemplates let you define reusable prompt skeletons with named placeholders.  
At runtime you fill in the placeholders with actual values.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Define a reusable prompt with a variable placeholder
prompt = ChatPromptTemplate.from_template(
    "You are an expert. Explain {topic} in simple terms."
)

# Format the prompt with actual value
formatted = prompt.format_messages(topic="neural networks")
print(formatted)

---
## Example 3 — Simple Chain (LCEL)

LCEL (LangChain Expression Language) uses the `|` pipe operator to chain components.  
This is the modern, recommended way to build pipelines in LangChain.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model='gpt-3.5-turbo', temperature=0)
prompt = ChatPromptTemplate.from_template("Explain {topic} briefly.")

# Build chain using pipe operator (LCEL)
chain = prompt | llm | StrOutputParser()

output = chain.invoke({"topic": "transformers in NLP"})
print(output)

---
## Example 4 — Agent with Tool

Agents dynamically decide which tools to call based on the query.  
Here we use a ReAct agent equipped with a DuckDuckGo web search tool.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_react_agent, AgentExecutor
from langchain_community.tools import DuckDuckGoSearchRun
from langchain import hub

llm = ChatOpenAI(model='gpt-3.5-turbo', temperature=0)
search_tool = DuckDuckGoSearchRun()   # web search capability
tools = [search_tool]

prompt = hub.pull("hwchase17/react")   # standard ReAct prompt
agent = create_react_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

result = executor.invoke({"input": "Who founded LangChain?"})
print(result["output"])

---
## Example 5 — Memory Example

ConversationBufferMemory stores the full chat history and injects it into every subsequent call.  
This allows the model to reference what was said earlier in the same conversation.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain

llm = ChatOpenAI(model='gpt-3.5-turbo', temperature=0)
# Memory stores the full conversation history
memory = ConversationBufferMemory()
conversation = ConversationChain(llm=llm, memory=memory, verbose=False)

r1 = conversation.predict(input="Hi! My name is Alice.")
r2 = conversation.predict(input="What is my name?")
print(r2)   # model correctly recalls 'Alice'

---
## Bonus — Custom Tool Definition

You can define your own tools using the `@tool` decorator.  
The docstring becomes the tool's description — this is what the agent reads to decide when to use it.

In [ ]:
from langchain.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiplies two numbers and returns the result."""
    return a * b

# The tool can now be passed to an agent or called directly
print(multiply.invoke({'a': 6, 'b': 7}))   # 42

---
## Bonus — Vector Store (RAG Foundation)

FAISS is a local, in-memory vector store great for development and prototyping.  
In production you'd swap it for Chroma, Pinecone, Weaviate, etc. — the API stays the same.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# Sample documents to index
docs = [Document(page_content="LangChain helps build LLM apps."),
        Document(page_content="Agents use tools to reason.")]

# Create FAISS vector store from documents
embeddings = OpenAIEmbeddings()
store = FAISS.from_documents(docs, embeddings)

results = store.similarity_search("What is LangChain?", k=1)
print(results[0].page_content)